### Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

1) A schema, including the name of the tool, a description, and/or argument definitions(often a JSON)
2) A function or coroutine to execute

In [2]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='<think>\nOkay, so I need to figure out why parrots talk. Let me start by recalling what I know about parrots. They\'re birds, right? And many species are known for mimicking human speech. But why do they do that? Is there an evolutionary reason? Maybe they mimic to communicate with each other or to adapt to their environment?\n\nFirst, I should consider the general behavior of parrots. Are there other animals that mimic sounds? Maybe other birds do it too, like mynas or some types of parrots. But why would mimicking human speech specifically be beneficial? Perhaps in the wild, they mimic other birds or environmental sounds to blend in or deceive predators. But in captivity, when they\'re around humans, they might learn to mimic human speech to interact with their human caregivers.\n\nI remember reading that parrots have a complex vocal apparatus, which allows them to produce a wide range of sounds. Their syrinx, the vocal organ in birds, is different from ours. Maybe

In [3]:
from langchain.tools import tool

@tool
def get_weather(location:str) -> str:
    """Get the weather of a particular location"""
    return f"It's sunny in {location}"


model_with_tools = model.bind_tools([get_weather])


In [6]:
response = model_with_tools.invoke("What the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Boston. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. The required parameter is location, which should be a string. Since the user mentioned Boston, I need to call get_weather with "Boston" as the location. I\'ll make sure to format the tool call correctly within the XML tags.\n', 'tool_calls': [{'id': 'qbq443aes', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 103, 'prompt_tokens': 154, 'total_tokens': 257, 'completion_time': 0.141406068, 'completion_tokens_details': {'reasoning_tokens': 79}, 'prompt_time': 0.006951058, 'prompt_tokens_details': None, 'queue_time': 0.159997403, 'total_time': 0.148357126}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand', 'finish_reason': 'tool_

### Tool execution loops

In [8]:
# Step 1: Model generates tool calls

messages = [{"role" : "user", "content":"What the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results

for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response

final_response = model_with_tools.invoke(messages)
print(final_response)


content='The weather in Boston is sunny.' additional_kwargs={'reasoning_content': "Okay, the user asked about the weather in Boston. I called the get_weather function with Boston as the location. The response came back saying it's sunny there. Now I need to relay that information clearly. Let me check if there's any additional details I should mention, but the function only provided the weather condition. So I'll just state that it's sunny in Boston. Keep it simple and direct.\n"} response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 191, 'total_tokens': 285, 'completion_time': 0.160833007, 'completion_tokens_details': {'reasoning_tokens': 82}, 'prompt_time': 0.009324168, 'prompt_tokens_details': None, 'queue_time': 0.052633162, 'total_time': 0.170157175}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019e841e-2759-7962-ad18-4f51d

In [9]:
messages

[{'role': 'user', 'content': 'What the weather in Boston?'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. I need to use the get_weather function. Let me check the function parameters. The required parameter is location, which should be a string. So I\'ll call get_weather with location set to "Boston". Make sure the JSON is correctly formatted with the name and arguments.\n', 'tool_calls': [{'id': 'mb6nbbp0t', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 90, 'prompt_tokens': 153, 'total_tokens': 243, 'completion_time': 0.125640408, 'completion_tokens_details': {'reasoning_tokens': 66}, 'prompt_time': 0.007674298, 'prompt_tokens_details': None, 'queue_time': 0.160624501, 'total_time': 0.133314706}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand', 'finish_reason